# MedSegDiff — Metrics vs Ground Truth (Water + Fat Fraction)

Computes per-muscle overlap metrics for MedSegDiff NPZ segmentations against
the myosegmenTUM ground truth.  Saves one CSV per muscle per modality to
`../results_water/` and `../results_fat_frac/`.

In [ ]:
import os, re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1
GT_BASE           = os.path.join('..', '..', 'myosegmenTUM')

# (output_key, gt_label_in_combined_gt, npz_key)
# GT label indices from medsegdiff/dataset.py GT_LABELS
MUSCLES = [
    ('R_gracilis',  5, 'R_gracilis'),
    ('L_gracilis',  1, 'L_gracilis'),
    ('R_sartorius', 8, 'R_sartorius'),
    ('L_sartorius', 4, 'L_sartorius'),
]

# (run_label, seg_dir, result_dir, csv_suffix)
# run_label is just for display; subject extraction always uses the modality
# token (WATER/FATFRACTION) present in the actual filename.
RUNS = [
    (
        'BOTH channels',
        os.path.join('..', 'both_channels'),
        os.path.join('..', 'results_channels'),
        'both',
    ),
    (
        'WATER',
        os.path.join('..', '..', 'medsegdiff_segs_water'),
        os.path.join('..', 'results_water'),
        'water',
    ),
    (
        'FATFRACTION',
        os.path.join('..', '..', 'medsegdiff_segs_fatfrac'),
        os.path.join('..', 'results_fat_frac'),
        'fatfrac',
    ),
]

for label, seg_dir, result_dir, _ in RUNS:
    os.makedirs(result_dir, exist_ok=True)
    n = len([f for f in os.listdir(seg_dir) if f.endswith('_medsegdiff.npz')]) \
        if os.path.isdir(seg_dir) else 0
    print(f'[{label}] {seg_dir}: {n} npz files  ->  {result_dir}')

In [ ]:
def evaluate_muscle(muscle_name, gt_label_idx, npz_key, seg_dir, result_dir, csv_suffix):
    npz_files = sorted(f for f in os.listdir(seg_dir) if f.endswith('_medsegdiff.npz'))
    rows = []

    for npz_file in npz_files:
        stem = npz_file.replace('_medsegdiff.npz', '')

        # Always parse subject/stack from the actual filename token (WATER or FATFRACTION),
        # regardless of the run label — this fixes the 'BOTH' case where filenames
        # still contain _WATER_ or _FATFRACTION_.
        m = re.match(r'(.+?)_(WATER|FATFRACTION)_stack(\d+)', stem)
        if not m:
            print(f'  could not parse: {npz_file}')
            continue
        subject   = m.group(1)
        stack_num = m.group(3)

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT missing: {gt_path}')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        npz_data = np.load(os.path.join(seg_dir, npz_file))
        pred_arr = npz_data[npz_key].astype(float)

        pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {npz_file}: empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        rows.append({
            'subject':                              subject,
            'stack':                                int(stack_num),
            'pred_file':                            npz_file,
            'gt_path':                              gt_path,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(rows)
    csv_path = os.path.join(result_dir,
                            f'df_{muscle_name}_medsegdiff_{csv_suffix}.csv')
    df.to_csv(csv_path)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df


all_dfs = {}

for run_label, seg_dir, result_dir, csv_suffix in RUNS:
    if not os.path.isdir(seg_dir):
        print(f'\n[skip] {seg_dir} not found')
        continue

    print(f'\n{"="*60}')
    print(f' {run_label}')
    print(f'{"="*60}')

    for muscle_name, gt_idx, npz_key in MUSCLES:
        print(f'\n── {muscle_name} ──')
        df = evaluate_muscle(muscle_name, gt_idx, npz_key,
                             seg_dir, result_dir, csv_suffix)
        all_dfs[(run_label, muscle_name)] = df

print('\nDone.')

In [ ]:
# Quick summary
for (modality, muscle), df in all_dfs.items():
    col = f'{muscle}_dice'
    if col in df.columns:
        print(f'{modality} {muscle}: mean Dice = {df[col].mean():.4f}  (n={len(df)})')